## Objective: To find if a person survived titanic drowning, given a set of features in tabular form

## Data Overview :
  - there are 11 features and the target column.
  - total 891 rows.
  - SibSp = number of siblings/Spouse on board for that person
  - Parch = number of parents/children on board for that person
  - rest of the features are self explanatory



In [ ]:
import numpy as np # linear algebra
import pandas as pd 
import os

# Reading data from kaggle input and storing them in dataframe variable train and test

In [ ]:
import pandas as pd

train = pd.read_csv(r"G:\MLA-C01-Materials\train.csv")
test = pd.read_csv(r"G:\MLA-C01-Materials\test.csv")
train.head()

In [ ]:
train.info()        # columns, data types, missing values 

# Exploratory Data Analysis:
## Asking questions about the dataset:
-----
- Who survived the most? male or female?
- Does class `Pclass` affect survival?
- Does family size matter?

## Steps in EDA:
 - Comparing Male vs female surviving counts and plotting them using seaborn's countplot function
 - Comparing how passenger class affected survival by using using seaborn's barplot function
 - finding missing values accroess the features and summing them 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.countplot(x='Survived', hue='Sex', data=train)
plt.title("Survival by Gender")
plt.show()

In [ ]:
sns.barplot(x='Pclass', y='Survived', data=train)
plt.title('Survival within passenger class')
plt.show()

# Now we will find the counts of missing values of duplicated entries


In [ ]:
train.isnull().sum()

## Age, cabin type and location of embarkment are the only features with missing values.
 - For `Age`, i will find the median of the column and fill the missing values with it.
 - For `Embarked`, i will find the mode of the column and use that to fill in the missing values
 - Since `Cabin`, is missing almost 80% of the data, i decided to drop that column altogether.

In [ ]:
train.fillna({
    "Age": train["Age"].median(),
    "Embarked": train["Embarked"].mode()[0]
}, inplace=True)

train.drop(columns=["Cabin"], inplace=True)

# Now we will convert categorical column `Sex` into dummy variables 

In [ ]:
train.head()

In [ ]:
train = pd.get_dummies(train, columns=['Sex'], drop_first = True)

In [ ]:
train['Sex_male']= train['Sex_male'].astype(int)

In [ ]:
train.head()

# We are going to do some feature engineering to help the model:
### Show that if having family members aboard or being alone was linked to survivability.
### we will find the correlation between `Survive` and  `isAlone` 

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch']+1
train['isAlone'] = (train['FamilySize']==1).astype(int)
train['isAlone'].corr(train['Survived'])

###  Since being alone is negatively correlated to the target variable, it acts as an important feature for the model to learn from

# Importing neccessary libraries for train/test split, model, evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Only selecting the numeric columns to train the model

In [ ]:
features = ["Pclass", "Age", "Fare", "Sex_male", "isAlone"]
X = train[features]
y = train["Survived"]

## Modelling :
### Splitting the *X* dataframe into training split `X_train` containing 80% of the data and validation split `X_val` containing the rest of the data.
-----
- Since its a categorical classification , we will be modelling the data using logistic regression for now. Later, more complex ensemble methods, boosting will be applied to show how that improves on the baseline results..

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# Evaluation :
- I am using Accuracy for now to evaluate the performance of the model. Accuracy calculates how much of the the model's prediction aligns with corresponding true values.
- After the model is fitted with `X_train` ,`y_train`, I stored the prection for `X_val` into a variable called `pred` which is a list with values *[0,1,0,1,0,0,0,1,1,0 etc]*
- Then, using the `accuracy_score` method, which compares the model's prediction stored in the `pred` variable with true values in the `y_val` variable, i generated a score and converted it into a percentage.

In [ ]:
pred = model.predict(X_val)
score = accuracy_score(y_val, pred)*100
print(f"Validation Accuracy:{score:.2f}%" )

# Using Random Forrest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rfc = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rfc.fit(X_train, y_train)
pred_rfc = rfc.predict(X_val)
score = accuracy_score(y_val, pred_rfc)*100
print(f"Random Forest Accuracy: {score:.2f}%")

In [ ]:
test['FamilySize'] = test['SibSp'] + train['Parch']+1
test['isAlone'] = (test['FamilySize']==1).astype(int)
test = pd.get_dummies(test, columns=['Sex'], drop_first = True)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rfc = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rfc.fit(X_train, y_train)
test_x = test[features]
pred_rfc = rfc.predict(test_x)
pred_rfc


In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": pred_rfc
})
submission.to_csv('my_submission.csv', index=False)